In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.collections as mcoll

plt.rcParams["font.family"] = "monospace"

_ROOT = Path.cwd().parent
_CORE = _ROOT / "core"
_DATA = _ROOT / "data"
EXPERT = _DATA / "expert.npz"

print(f"NumPy Version: {np.__version__}")

In [ ]:
print(f"Data Path: {EXPERT}")
dExpert = np.load(EXPERT)

print("Expert Data Keys:", dExpert.files)

# Expert Data
Expert data consists of data in shape `(20, 500, 8)`
- `20`: Number of waypoints
- `500`: Steps per waypoint
- `8`: The state and action vector for each step

In [ ]:
numOfTraj, stepsPerTraj, params = dExpert["trajectories"].shape

print(
    f"Number of Trajectories: {numOfTraj}",
    f"Steps per Trajectory: {stepsPerTraj}",
    f"Parameters per Step: {params}",
    sep="\n",
)

In [ ]:
dConcat = np.concatenate(dExpert["trajectories"], axis=0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

plt.clf()
plt.close('all')

fig, ax = plt.subplots(figsize=(10, 6))

# Determine number of trajectories to animate
numToAnimate = min(len(dExpert["trajectories"]), 10)

def update(frame):
     # Clear the axis for the next frame
    ax.clear()
    
    # Setup plot aesthetics for each frame
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.invert_yaxis()
    ax.grid(True, linestyle='--', alpha=0.3, zorder=0)
    ax.set_title(f"Expert Trajectory: {frame + 1}")
    ax.set_xlabel("X Position")
    ax.set_ylabel("Y Position")

    # Extract current trajectory
    traj = dExpert["trajectories"][frame]
    X = traj[:, 0]
    Y = traj[:, 1]
    
    points = np.array([X, Y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    
    norm = plt.Normalize(vmin=0, vmax=len(X) - 1)
    segColors = np.linspace(0, len(X) - 1, len(X) - 1)
    
    lc = mcoll.LineCollection(segments, cmap='managua', norm=norm, alpha=0.7)
    lc.set_array(segColors)
    ax.add_collection(lc)
    
    stepIdx = np.arange(0, len(X), 10)
    ax.scatter(
        X[::10], Y[::10],
        c=stepIdx, cmap='managua', norm=norm,
        s=20, marker="o", linewidths=0.5, zorder=5
    )
    
    sColor = plt.cm.managua(norm(0))
    eColor = plt.cm.managua(norm(len(X) - 1))
    
    ax.text(X[0], Y[0], " Start", fontsize=9, color=sColor, weight="bold", 
            ha="right", va="bottom")
    ax.text(X[-1], Y[-1], " End", fontsize=9, color=eColor, weight="bold", 
            ha="left", va="top")

    plt.tight_layout()

# Create the animation
ani = FuncAnimation(fig, update, frames=numToAnimate, interval=1000)

# Save the animation
ani.save("trajectories.gif", writer='pillow', dpi=900)

plt.show()

# Plotting the Parameters

For a single chosen trajectory, we can plot the action and state vectors

In [ ]:
trajIdx = np.random.randint(numOfTraj)
traj = dExpert["trajectories"][trajIdx]

timeSteps = np.arange(stepsPerTraj)

In [ ]:
# Name the parameters
#  state = np.array([x, y, vx, vy, theta, omega, windX, windY], dtype=np.float32)
#  action = np.array([lThrust, rThrust], dtype=np.float32)
x, y, vx, vy, theta, omega, windX, windY = traj[:, :8].T
lThrust, rThrust = traj[:, 8:].T

# Compute wind magnitude
windMag = np.hypot(windX, windY)

In [ ]:
NAVY: str = "#223D5A"    # Deep Navy/Teal (Start of spectrum)
SEAFOAM: str = "#2B7371"   # Muted Seafoam/SEAFOAM
SAGE: str = "#769371"   # Sage (The "transition" SAGE-ish/neutral zone)
OCHRE: str = "#BBA358"  # Ochre/Goldenrod
GOLD: str = "#E9C172"     # Dusty Terracotta/Gold
SAND: str = "#FEF0BD"  # Pale Sand (End of spectrum)

In [ ]:
plt.figure(figsize=(12, 8))
# ------------ Velocity Plot ------------ 
plt.plot(timeSteps, vx, label=r"X Velocity ($v_X$)", color=NAVY)
plt.plot(timeSteps, vy, label=r"Y Velocity ($v_Y$)", color=OCHRE)
plt.title(r"Velocity ($v_X,\ v_Y$)", fontsize=16)
plt.xlabel("Time Step")
plt.ylabel("Velocity")
plt.legend(loc="upper right", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
# ------------ Orientation & Angular Velocity Plot ------------
plt.plot(timeSteps, theta, label=r"Orientation ($\theta$)", color=NAVY)
plt.plot(timeSteps, omega, label=r"Angular Velocity ($\omega$)", color=OCHRE)
plt.title(r"Orientation ($\theta$) | Angular Velocity ($\omega$)", fontsize=16)
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.ylim(-0.5, 0.8)
plt.legend(loc="upper right", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
# ------------ Actions (Thrust) Plot ------------
plt.plot(timeSteps, lThrust, label=r"Left Thrust ($T_{L}$)", color=NAVY)
plt.plot(timeSteps, rThrust, label=r"Right Thrust ($T_{R}$)", color=OCHRE)
plt.title(r"Actions ($T_{Left}$, $T_{Right}$)", fontsize=16)
plt.xlabel("Time Step")
plt.ylabel("Thrust")
plt.legend(loc="upper right", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

# Animating the Parameters

It would be a great practice to visualize the trajectory and the parameters in real-time

In [ ]:
import numpy as np
import math
import warnings
from dataclasses import dataclass

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.collections as mcoll
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Embedding videos in Jupyter Notebooks
from IPython.display import Video

mpl.rcParams['animation.embed_limit'] = 100.0
warnings.filterwarnings("ignore", category=UserWarning)

@dataclass
class Screen:
    WIDTH: int = 1200
    HEIGHT: int = 800
    MARGIN: int = 20

trajIdx = np.random.randint(numOfTraj)
traj = dExpert["trajectories"][trajIdx]

X = traj[:, 0]
Y = traj[:, 1]

numFrames = len(X)
segColors = np.linspace(0, numFrames - 1, numFrames - 1)
norm = plt.Normalize(vmin=0, vmax=numFrames - 1)

In [ ]:
# Drone Visualization
lARM: int = 30
sTHRUST: float = .5

def _thrustColors(thrust):
    """Maps thrust [0, 1] to the Managua spectrum."""
    thrust = np.clip(thrust, 0, 1)
    return plt.cm.managua(thrust)

In [ ]:
# ------------------------------------------------ Setup Figure and Grid ------------------------------------------------
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(4, 2, width_ratios=[1.2, 1])

# Left: Trajectory Plot
axTraj = fig.add_subplot(gs[:, 0])
axTraj.set_title("Expert Trajectory")
axTraj.set_xlim(0, Screen.WIDTH)
axTraj.set_ylim(Screen.HEIGHT, 0)
axTraj.grid(True, linestyle='--', alpha=0.3, zorder=0)

# ------------------------------------------------ Wind HUD ------------------------------------------------
RUST: str = "#C65D47"
CRIMSON: str = "#A63D40"

axWindHUD = inset_axes(
    axTraj,
    width="12%",
    height="12%",
    loc="upper right",
    borderpad=0.8
)

axWindHUD.set_facecolor((0, 0, 0, 0.0))

windLimit = max(
    np.max(np.abs(windX)),
    np.max(np.abs(windY))
) * 1.15

axWindHUD.set_xlim(-windLimit, windLimit)
axWindHUD.set_ylim(-windLimit, windLimit)

# Crosshair
axWindHUD.axhline(0, color=CRIMSON, lw=0.8, alpha=0.2)
axWindHUD.axvline(0, color=CRIMSON, lw=0.8, alpha=0.2)

# Remove clutter
axWindHUD.set_xticks([])
axWindHUD.set_yticks([])
axWindHUD.set_title("Wind Vector", fontsize=8, color=CRIMSON, pad=-5)

for spine in axWindHUD.spines.values():
    spine.set_color(CRIMSON)
    spine.set_alpha(0.)

# Single dynamic wind vector
windArrow = axWindHUD.quiver(
    [0],
    [0],
    [0],
    [0],
    angles='xy',
    scale_units='xy',
    scale=1,
    color=CRIMSON,
    width=0.025,
    zorder=5
)

# Wind magnitude label
windText = axWindHUD.text(
    0.5,
    0.05,
    "",
    transform=axWindHUD.transAxes,
    fontsize=8,
    color=CRIMSON,
    ha='center'
)

# Right: Parameter Subplots
axVel = fig.add_subplot(gs[0, 1])
axOrient = fig.add_subplot(gs[1, 1])
axThr = fig.add_subplot(gs[2, 1])

for ax in [axVel, axOrient, axThr]:
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.set_xlim(timeSteps[0], timeSteps[-1])

# ---- PLOTS: Parameters ----
axVel.set_title(r"Velocity ($v_X,\ v_Y$)")
axVel.set_ylabel("Velocity")
axVel.grid(True, linestyle='--', alpha=0.3)
axVel.set_xlim(timeSteps[0], timeSteps[-1])
axVel.set_ylim(min(np.min(vx), np.min(vy)) - 0.1, max(np.max(vx), np.max(vy)) + 0.1)

axOrient.set_title(r"Orientation ($\theta$) | Angular Velocity ($\omega$)")
axOrient.set_ylabel("Value")
axOrient.grid(True, linestyle='--', alpha=0.3)
axOrient.set_xlim(timeSteps[0], timeSteps[-1])
axOrient.set_ylim(-1.5, 1.8)

axThr.set_title(r"Actions ($T_{Left}$, $T_{Right}$)")
axThr.set_xlabel("Time Step")
axThr.set_ylabel("Thrust")
axThr.grid(True, linestyle='--', alpha=0.3)
axThr.set_xlim(timeSteps[0], timeSteps[-1])
axThr.set_ylim(min(np.min(lThrust), np.min(rThrust)) - 0.1, max(np.max(lThrust), np.max(rThrust)) + 0.1)

axWind = fig.add_subplot(gs[3, 1])
axWind.set_title("Wind Magnitude")
axWind.set_xlabel("Time Step")
axWind.set_ylabel("|W|")
axWind.set_ylim(0, np.max(windMag) * 1.15)
axWind.set_xlim(timeSteps[0], timeSteps[-1])
axWind.grid(True, linestyle='--', alpha=0.3)

# ------------------------------------------------ Initialize Artists ------------------------------------------------
lc = mcoll.LineCollection([], cmap='managua', norm=norm, alpha=0.7, linewidth=4)
axTraj.add_collection(lc)
scatter = axTraj.scatter([], [], c=[], cmap='managua', norm=norm, s=0, zorder=5)
# Add wind quiver (arrows)
sampleStep = 12
maxArrows = len(np.arange(0, numFrames, sampleStep))

# Body Frame
droneFrame, = axTraj.plot([], [], color=SEAFOAM, lw=4, solid_capstyle='round', zorder=10)

# Visualizing Rotors
lRotor, = axTraj.plot([], [], 'o', markersize=10, markeredgecolor='white', markeredgewidth=0.5, zorder=11)
rRotor, = axTraj.plot([], [], 'o', markersize=10, markeredgecolor='white', markeredgewidth=0.5, zorder=11)

# Parameter Lines
lineVx, = axVel.plot([], [], color=NAVY)
lineVy, = axVel.plot([], [], color=OCHRE)
lineTheta, = axOrient.plot([], [], color=NAVY)
lineOmega, = axOrient.plot([], [], color=OCHRE)
lineLeftThr, = axThr.plot([], [], color=NAVY)
lineRightThr, = axThr.plot([], [], color=OCHRE)
lineWindMag, = axWind.plot([], [], color=CRIMSON)

def update(frame):
    if frame < 2:
        return lc, scatter, droneFrame, lRotor, rRotor, lineVx, lineVy, lineTheta, lineOmega, lineLeftThr, lineRightThr

    # Update Trajectory Path
    points = np.array([X[:frame], Y[:frame]]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc.set_segments(segments)
    lc.set_array(segColors[:frame-1])
    
    currWindX = windX[frame - 1]
    currWindY = windY[frame - 1]

    windArrow.set_UVC(
        [currWindX],
        [currWindY]
    )

    currWindMag = np.hypot(currWindX, currWindY)

    windText.set_text(
        f"|W| = {currWindMag:.1f}"
    )

    # Decimated Scatter Trail
    maskStep = np.arange(0, frame, 10)
    if len(maskStep) > 0:
        scatter.set_offsets(np.c_[X[maskStep], Y[maskStep]])
        scatter.set_array(maskStep)

# ------------------------------------------------ Drone Rendering ------------------------------------------------
    currX, currY = X[frame-1], Y[frame-1]
    currTheta = theta[frame-1]
    
    dx = lARM * math.cos(currTheta)
    dy = lARM * math.sin(currTheta)
    
    lx, ly = currX - dx, currY - dy
    rx, ry = currX + dx, currY + dy
    
    # Update Body
    droneFrame.set_data([lx, rx], [ly, ry])
    
    # Update Rotor Dots (Positions)
    lRotor.set_data([lx], [ly])
    rRotor.set_data([rx], [ry])
    
    # Update Rotor Colors (Thrust Intensity)
    lAction, rAction = lThrust[frame-1], rThrust[frame-1]
    lRotor.set_markerfacecolor(_thrustColors(lAction))
    lRotor.set_markeredgecolor(_thrustColors(lAction))

    rRotor.set_markerfacecolor(_thrustColors(rAction))
    rRotor.set_markeredgecolor(_thrustColors(rAction))

    droneFrame.set_color(_thrustColors((lAction + rAction) / 2))

    # Update Parameter Plots
    lineVx.set_data(timeSteps[:frame], vx[:frame])
    lineVy.set_data(timeSteps[:frame], vy[:frame])
    lineTheta.set_data(timeSteps[:frame], theta[:frame])
    lineOmega.set_data(timeSteps[:frame], omega[:frame])
    lineLeftThr.set_data(timeSteps[:frame], lThrust[:frame])
    lineRightThr.set_data(timeSteps[:frame], rThrust[:frame])
    lineWindMag.set_data(
        timeSteps[:frame],
        windMag[:frame]
    )

    return (lc, scatter, droneFrame, lRotor, rRotor, 
            lineVx, lineVy, lineTheta, lineOmega, 
            lineLeftThr, lineRightThr, 
            lineWindMag, 
            windArrow, windText)

# ------------------------------------------------ Generate and Save ------------------------------------------------
plt.tight_layout()

anim = animation.FuncAnimation(fig, update, frames=numFrames, interval=15, blit=True)
anim.save(f'exTrajDrone[{trajIdx}].mp4', writer='ffmpeg', fps=60, progress_callback=lambda i, n: print(f"Saving Frame {i + 1}/{n}", end='\r'))
print(f"\nAnimation 'exTrajDrone[{trajIdx}].mp4' saved successfully!")

In [ ]:
Video(f"exTrajDrone[{trajIdx}].mp4", embed=True)